# 🔬 YOLO-GLAAM Cataract Detection Training

**Run this notebook on Google Colab with GPU runtime.**

Prerequisites:
1. Upload your project folder to Google Drive as `cataract_detection/`
2. Upload dataset to `cataract_detection/data/raw/slitlamp/`

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set project path
PROJECT_PATH = '/content/drive/MyDrive/cataract_detection'
print(f'Project path: {PROJECT_PATH}')

## Step 2: Verify GPU

In [ ]:
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Step 3: Install Dependencies

In [ ]:
%cd {PROJECT_PATH}
!pip install -q ultralytics albumentations grad-cam tensorboard

## Step 4: Verify Project Structure

In [ ]:
import os

# Check project files
print('Project files:')
!ls -la {PROJECT_PATH}

print('\nModels directory:')
!ls -la {PROJECT_PATH}/models

print('\nDataset:')
!ls {PROJECT_PATH}/data/raw/slitlamp/ 2>/dev/null | head -5 || echo 'Dataset not found! Please upload to data/raw/slitlamp/'

## Step 5: Train YOLO Lens Detector (Optional)

Only run this if you have annotated lens bounding boxes.

In [ ]:
# Uncomment to train YOLO (only if you have annotations)
# !python training/train_yolo.py --data_yaml data/lens.yaml --epochs 50

## Step 6: Train Hybrid Model (Main Training)

In [ ]:
%cd {PROJECT_PATH}

!python training/train_hybrid.py \
    --data_root data/raw/slitlamp \
    --epochs 100 \
    --batch_size 16 \
    --backbone mobilenetv2 \
    --attention glaam \
    --checkpoint_dir checkpoints

## Step 7: View Training Logs (TensorBoard)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {PROJECT_PATH}/logs

## Step 8: Verify Checkpoint Saved

In [ ]:
print('Saved checkpoints:')
!ls -la {PROJECT_PATH}/checkpoints/

# Check file size
!du -h {PROJECT_PATH}/checkpoints/*.pth 2>/dev/null || echo 'No checkpoints found'

## Step 9: Quick Test Inference

In [ ]:
import sys
sys.path.insert(0, PROJECT_PATH)

import torch
from models import HybridCataractModel

# Load trained model
model = HybridCataractModel(
    backbone='mobilenetv2',
    attention_type='glaam',
    use_yolo=False
)

checkpoint = torch.load(f'{PROJECT_PATH}/checkpoints/best.pth')
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded model from epoch {checkpoint['epoch']}")
print(f"Best validation accuracy: {checkpoint['best_val_acc']:.4f}")

## ✅ Done!

Your trained model is saved to:
- `Google Drive/cataract_detection/checkpoints/best.pth`

Download this file to your local PC for inference!